# Drawing and Geometric Transformations

> **Beginner · Geometry**


## Why this matters

Annotations make pipeline output inspectable, while transforms normalize images before measurement or recognition. Both depend on getting coordinates right.

**Where it appears:** Bounding-box displays, image registration, document rectification, data augmentation, and augmented overlays.


## Learning Objectives

- Draw shapes, text, and annotations with correct parameters (thickness, line type, anchor points)
- Build a reusable annotation function for detection-style bounding boxes and labels
- Understand anti-aliasing (`cv2.LINE_AA`) and when to use it
- Apply resizing, rotation, translation, and flipping correctly
- Understand affine vs perspective transforms and when each applies
- Build a reusable 'rotate about arbitrary point without cropping' function


## Prerequisites

05 Pixels, Channels, Color, and Masks

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.line`, `cv2.rectangle`, `cv2.putText`, `cv2.resize`, `cv2.warpAffine`, `cv2.warpPerspective`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Drawing Functions

Drawing functions are used constantly downstream -- to visualize detections,
contours, keypoints, and tracking results. They all draw **in place** by
default (mutating the array), which frequently surprises beginners who
then can't recover the original image. This notebook establishes the
"draw on a copy" convention used in every visualization from here on.


### Geometric Transformations

Geometric transforms are represented as matrices applied via
`cv2.warpAffine` (2x3, preserves parallel lines: rotation, scale,
translation, shear) or `cv2.warpPerspective` (3x3, can map any quadrilateral
to any other -- used for document/plane rectification). A common beginner
bug is rotating an image and having corners get cut off; fixing this
requires recomputing the output canvas size, not just the rotation matrix.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Drawing Functions


### 1. Basic shapes and the in-place mutation gotcha

`cv2.rectangle`, `cv2.circle`, `cv2.line` modify the array in place and also return it -- always draw on a `.copy()` if you need the original later.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def draw_demo_shapes(image: np.ndarray) -> np.ndarray:
    """Draws on a COPY of `image` and returns the annotated copy, leaving the input untouched."""
    canvas = image.copy()
    cv2.rectangle(canvas, (20, 20), (140, 100), (0, 0, 255), thickness=2)
    cv2.circle(canvas, (250, 60), 40, (0, 255, 0), thickness=-1)  # -1 = filled
    cv2.line(
        canvas, (0, 150), (400, 150), (255, 0, 0), thickness=2, lineType=cv2.LINE_AA
    )
    return canvas


base = cv2.resize(load_real_image("images/objects", "geometric_shapes.png"), (500, 400))
annotated = draw_demo_shapes(base)

print(
    "Original untouched:",
    np.array_equal(
        base,
        cv2.resize(
            load_real_image("images/objects", "geometric_shapes.png"), (500, 400)
        ),
    ),
)
show_grid([("original", base), ("annotated copy", annotated)])

### 2. Text annotation with a readable helper

`cv2.putText` requires manually managing font, scale, thickness, and the *bottom-left* anchor point -- wrap it so callers just supply text and a position.


In [ ]:
def label(
    image: np.ndarray,
    text: str,
    top_left: tuple[int, int],
    color=(255, 255, 255),
    bg_color=(0, 0, 0),
) -> np.ndarray:
    """Draw `text` with a filled background box for readability -- the standard
    'detection label' look used in the object-tracking and detection notebooks."""
    canvas = image.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness = 0.8, 2
    (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
    x, y = top_left
    cv2.rectangle(canvas, (x, y - th - baseline), (x + tw, y + baseline), bg_color, -1)
    cv2.putText(canvas, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)
    return canvas


labeled = label(annotated, "shape id: 1", (20, 130))
show_grid([("labeled", labeled)], cols=1)

### 3. A reusable bounding-box annotator

Combine rectangle + label into one function -- this exact utility is reused, unmodified in spirit, in the tracking, detection, and face-recognition notebooks.


In [ ]:
def draw_detection(
    image: np.ndarray, box: tuple[int, int, int, int], text: str, color=(0, 200, 255)
) -> np.ndarray:
    """box = (x1, y1, x2, y2). Draws a bounding box with a text label above it."""
    x1, y1, x2, y2 = box
    canvas = image.copy()
    cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 3)
    canvas = label(canvas, text, (x1, max(y1 - 6, 12)), bg_color=color, color=(0, 0, 0))
    return canvas


result = draw_detection(base, (72, 153, 182, 259), "triangle 0.94")
show_grid([("detection-style annotation", result)], cols=1)

## Part 2: Geometric Transformations


### 1. Resize, flip, translate

The basics: `cv2.resize` (watch interpolation choice), `cv2.flip`, and translation via a manually built affine matrix.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def translate(image: np.ndarray, dx: int, dy: int) -> np.ndarray:
    """Shift an image by (dx, dy) pixels using an affine matrix."""
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))


scene = load_real_image("images/standard", "billboard.png")
resized = cv2.resize(scene, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
flipped = cv2.flip(scene, 1)  # 1 = horizontal
shifted = translate(scene, 40, -20)

show_grid(
    [
        ("original", scene),
        ("resized 0.5x", resized),
        ("flipped", flipped),
        ("translated", shifted),
    ]
)

### 2. Rotation without cropping corners

`cv2.getRotationMatrix2D` rotates about a center but keeps the ORIGINAL canvas size, silently cropping corners. Fix it by expanding the canvas to fit the rotated bounding box.


In [ ]:
def rotate_no_crop(image: np.ndarray, angle_deg: float) -> np.ndarray:
    """Rotate `image` by `angle_deg` about its center, expanding the canvas so nothing is cropped."""
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, scale=1.0)

    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)

    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    return cv2.warpAffine(image, M, (new_w, new_h))


naive = cv2.warpAffine(
    scene,
    cv2.getRotationMatrix2D((scene.shape[1] / 2, scene.shape[0] / 2), 35, 1.0),
    (scene.shape[1], scene.shape[0]),
)
fixed = rotate_no_crop(scene, 35)

show_grid([("naive rotation (corners cropped)", naive), ("rotate_no_crop", fixed)])

### 3. Perspective transform for plane rectification

Given 4 source points and 4 destination points, `cv2.getPerspectiveTransform` computes the homography to 'flatten' a tilted quadrilateral -- e.g. a photographed document -- into a fronto-parallel view.


In [ ]:
def rectify_quad(
    image: np.ndarray, src_pts: np.ndarray, out_size=(300, 400)
) -> np.ndarray:
    """Warp the quadrilateral `src_pts` (4x2, TL,TR,BR,BL order) to fill a flat rectangle."""
    w, h = out_size
    dst_pts = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    M = cv2.getPerspectiveTransform(src_pts.astype(np.float32), dst_pts)
    return cv2.warpPerspective(image, M, (w, h))


# Simulate a tilted "document" quadrilateral on the canvas
doc = np.full((300, 300, 3), 255, dtype=np.uint8)
skewed_corners = np.float32([[40, 20], [260, 60], [230, 280], [10, 250]])
cv2.polylines(doc, [skewed_corners.astype(np.int32)], True, (0, 0, 0), 2)
cv2.putText(doc, "DOC", (100, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)

rectified = rectify_quad(doc, skewed_corners)
show_grid([("tilted 'document'", doc), ("rectified", rectified)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Drawing Functions: Heads-Up Display (HUD) Gauge Overlay

In robotics and smart mobility pipelines, visualizing system state (radar sweeps, gauges, telemetry data) on raw video frames is a standard dashboard task. Here, we build an interactive HUD template utilizing OpenCV's drawing functions.


In [ ]:
hud_canvas = np.zeros((300, 300, 3), dtype=np.uint8)

# Draw central target radar circles
center = (150, 150)
cv2.circle(hud_canvas, center, 120, (0, 255, 0), 1)
cv2.circle(hud_canvas, center, 80, (0, 255, 0), 1)
cv2.circle(hud_canvas, center, 40, (0, 255, 0), 1)

# Draw a radar sweeping ray
import math

angle = 45  # Degrees
rad = math.radians(angle)
endpoint = (int(150 + 120 * math.cos(rad)), int(150 - 120 * math.sin(rad)))
cv2.line(hud_canvas, center, endpoint, (0, 255, 200), 2)

# Draw indicator tick marks
cv2.line(hud_canvas, (150, 20), (150, 30), (0, 255, 0), 2)
cv2.line(hud_canvas, (150, 270), (150, 280), (0, 255, 0), 2)

# Add numeric telemetry overlays
cv2.putText(
    hud_canvas, "RANGE: 12.4m", (10, 280), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1
)
cv2.putText(
    hud_canvas, "SYS: ACTIVE", (180, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1
)

print("HUD Canvas generated successfully.")
show(hud_canvas, "Interactive HUD Radar Dashboard")

### Mini Project — Geometric Transformations: Virtual Billboard Homography Projection

A classic Augmented Reality feature is placing a custom logo or image onto a target planar surface (such as a TV screen or billboard) in a background image. This is accomplished using Homography matrices to warp coordinates between image spaces.


In [ ]:
# Source image to overlay (a blue logo)
logo = np.zeros((100, 100, 3), dtype=np.uint8)
logo[:, :] = [255, 0, 0]  # Blue logo background
cv2.rectangle(logo, (20, 20), (80, 80), (255, 255, 255), 3)  # White interior square

# Destination scene
scene = load_real_image("images/standard", "billboard.png")
sh, sw = scene.shape[:2]

# Define 4 source corner coordinates
src_pts = np.array([[0, 0], [99, 0], [99, 99], [0, 99]], dtype=np.float32)

# Define 4 target points on the scene to project logo into (forming a skewed quad)
dst_pts = np.array([[250, 300], [800, 250], [800, 600], [250, 700]], dtype=np.float32)

# Compute Homography matrix
H, mask = cv2.findHomography(src_pts, dst_pts)

# Warp source image onto destination coordinates
warped_logo = cv2.warpPerspective(logo, H, (sw, sh))

# Mask out destination pixels and blend
logo_mask = cv2.warpPerspective(np.ones((100, 100), dtype=np.uint8) * 255, H, (sw, sh))
scene_bg = cv2.bitwise_and(scene, scene, mask=cv2.bitwise_not(logo_mask))
composited_scene = cv2.add(scene_bg, warped_logo)

print("Billboard projection completed.")
show(composited_scene, "Virtual Billboard Overlay")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Drawing Functions
1. Write `draw_multiple_detections(image, boxes_and_labels)` that loops `draw_detection` over a list.
2. Add polygon drawing (`cv2.polylines`) support to a new helper for arbitrary-shaped outlines.
3. Modify `label` to auto-choose black or white text color based on background brightness for contrast.

Use the empty cell below to work through them.


#### Solutions — Drawing Functions

In [ ]:
# Solution 1: draw_multiple_detections
def draw_multiple_detections(
    image: np.ndarray, boxes_and_labels: list[tuple[tuple[int, int, int, int], str]]
) -> np.ndarray:
    """Draw bounding boxes and labels for multiple detections."""
    canvas = image.copy()
    for bbox, label_str in boxes_and_labels:
        x, y, w, h = bbox
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(
            canvas,
            label_str,
            (x, max(y - 5, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            1,
        )
    return canvas

In [ ]:
# Solution 2: Polygon outlines drawing support using cv2.polylines
def draw_polygon_outlines(
    image: np.ndarray, points: np.ndarray, color=(255, 0, 0), thickness=2
) -> np.ndarray:
    """Draw an arbitrary polygon outline using cv2.polylines."""
    canvas = image.copy()
    # Reshape points to match cv2 structure requirement: [1, n_pts, 2]
    pts = points.reshape((-1, 1, 2))
    cv2.polylines(canvas, [pts], isClosed=True, color=color, thickness=thickness)
    return canvas

In [ ]:
# Solution 3: Contrast-aware text color selection based on brightness
def draw_contrast_label(
    image: np.ndarray, text: str, pos: tuple[int, int]
) -> np.ndarray:
    """Draw text with auto-chosen color (black/white) based on local patch brightness."""
    canvas = image.copy()
    x, y = pos

    # Crop a small region of interest to evaluate average brightness
    h, w = image.shape[:2]
    patch = image[max(0, y - 10) : min(h, y + 10), max(0, x) : min(w, x + 100)]

    # Calculate average intensity
    gray_patch = (
        cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY) if patch.size > 0 else np.zeros((1, 1))
    )
    mean_val = np.mean(gray_patch)

    # If background is bright, use black text; otherwise use white text
    text_color = (0, 0, 0) if mean_val > 127 else (255, 255, 255)
    cv2.putText(canvas, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)
    return canvas


# Test code execution
img = cv2.resize(load_real_image("images/objects", "geometric_shapes.png"), (500, 400))
boxes = [((200, 173, 92, 84), "circle 0.88"), ((325, 172, 104, 91), "square 0.91")]
res = draw_multiple_detections(img, boxes)
poly_pts = np.array([[50, 10], [100, 50], [80, 120], [20, 100]], dtype=np.int32)
res_poly = draw_polygon_outlines(img, poly_pts)
res_contrast = draw_contrast_label(img, "TEST LABEL", (100, 150))
print("Drawing operations verified.")

### Exercises — Geometric Transformations
1. Write `scale_about_point(image, cx, cy, factor)` that scales an image about an arbitrary point, not just the center.
2. Extend `rotate_no_crop` to accept a scale factor as well as an angle.
3. Use `cv2.findHomography` with more than 4 (noisy) point correspondences instead of exactly 4 -- compare robustness.

Use the empty cell below to work through them.


#### Solutions — Geometric Transformations

In [ ]:
# Solution 1: scale_about_point
def scale_about_point(image: np.ndarray, cx: int, cy: int, factor: float) -> np.ndarray:
    """Scale an image relative to an arbitrary center coordinate (cx, cy)."""
    # T1: Translate center to origin
    T1 = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float32)
    # S: Scale
    S = np.array([[factor, 0, 0], [0, factor, 0], [0, 0, 1]], dtype=np.float32)
    # T2: Translate back
    T2 = np.array([[1, 0, cx], [0, 1, cy], [0, 0, 1]], dtype=np.float32)

    # Combined transform matrix
    M_full = T2 @ S @ T1
    # Crop to 2x3 affine format
    M_affine = M_full[:2, :]

    h, w = image.shape[:2]
    return cv2.warpAffine(image, M_affine, (w, h))

In [ ]:
# Solution 2: rotate_no_crop with scale factor
def rotate_no_crop(image: np.ndarray, angle: float, scale: float = 1.0) -> np.ndarray:
    """Rotate an image without cropping margins, optionally scaling."""
    h, w = image.shape[:2]
    cx, cy = w / 2, h / 2

    # Create baseline rotation matrix
    M = cv2.getRotationMatrix2D((cx, cy), angle, scale)

    # Compute new boundary dimensions
    cos_val = np.abs(M[0, 0])
    sin_val = np.abs(M[0, 1])

    new_w = int((h * sin_val) + (w * cos_val))
    new_h = int((h * cos_val) + (w * sin_val))

    # Update rotation translation components to avoid translation clipping
    M[0, 2] += (new_w / 2) - cx
    M[1, 2] += (new_h / 2) - cy

    return cv2.warpAffine(image, M, (new_w, new_h))

In [ ]:
# Solution 3: cv2.findHomography with noisy point correspondences
def test_homography_ransac() -> None:
    """Compare plain Homography vs RANSAC Homography on noisy coordinates."""
    src = np.array([[0, 0], [100, 0], [100, 100], [0, 100], [50, 50]], dtype=np.float32)
    dst = src.copy()

    # Corrupt last coordinate heavily to simulate detection outlier
    dst[4] += [200.0, 200.0]

    # 1. Plain least-squares
    H_ls, _ = cv2.findHomography(src, dst, method=0)
    # 2. RANSAC outlier rejection
    H_ransac, _ = cv2.findHomography(src, dst, method=cv2.RANSAC)

    print(
        "Plain Least-Squares H projection error for (0,0):",
        (H_ls @ np.array([0, 0, 1]))[:2],
    )
    print("RANSAC H projection error for (0,0):", (H_ransac @ np.array([0, 0, 1]))[:2])


# Test transforms
img = load_real_image("images/standard", "billboard.png")
res1 = scale_about_point(img, 100, 100, 1.5)
res2 = rotate_no_crop(img, 30, 0.8)
test_homography_ransac()

## Summary

You can annotate images clearly and apply resize, affine, and perspective transforms without losing track of coordinate conventions.

- **Best Practices:** Copy before drawing when the original matters, choose interpolation for the data type, and validate point order before a perspective warp.
- **Common Pitfalls:** In-place drawing surprises, reversed width/height, clipped rotations, and incorrectly ordered quadrilateral points.